<a href="https://colab.research.google.com/github/kratagyapandey/PYTHON-MAJOR-PROJECT-/blob/main/Student_Management_and_Performance_Analytical_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os

FILE_NAME = "students.json"
MAX_MARKS = 100
MAX_SUBJECTS = 15


# input lene ke functions

def get_text(prompt):
    while True:
        value = input(prompt).strip()
        if value:
            return value
        print("[!] This field cannot be blank.")


def get_int(prompt, min_value, max_value):
    while True:
        try:
            value = int(input(prompt).strip())
        except ValueError:
            print("[!] Please enter a whole number.")
            continue

        if min_value <= value <= max_value:
            return value
        print(f"[!] Enter a number between {min_value} and {max_value}.")


def get_float(prompt, min_value, max_value):
    while True:
        try:
            value = float(input(prompt).strip())
        except ValueError:
            print("[!] Please enter a valid number.")
            continue

        if min_value <= value <= max_value:
            return value
        print(f"[!] Enter a value between {min_value} and {max_value}.")


def get_marks():
    num_subjects = get_int(f"Number of subjects (1-{MAX_SUBJECTS}): ", 1, MAX_SUBJECTS)
    marks = []
    for i in range(1, num_subjects + 1):
        m = get_float(f"  Subject {i} marks (0-{MAX_MARKS}): ", 0, MAX_MARKS)
        marks.append(m)
    return marks


class Student:

    def __init__(self, roll_no, name, age, course, marks):
        self.roll_no = roll_no
        self.name = name
        self.age = age
        self.course = course
        self.marks = marks

    def calculate_total(self):
        return sum(self.marks)

    def calculate_average(self):
        if not self.marks:
            return 0.0
        return self.calculate_total() / len(self.marks)

    def calculate_grade(self):
        avg = self.calculate_average()
        if avg >= 90:
            return "A+"
        elif avg >= 80:
            return "A"
        elif avg >= 70:
            return "B"
        elif avg >= 60:
            return "C"
        elif avg >= 50:
            return "D"
        else:
            return "F (Fail)"

    def display_report(self):
        marks_text = ", ".join(f"{m:g}" for m in self.marks)

        print("\n" + "=" * 50)
        print("            STUDENT REPORT CARD")
        print("=" * 50)
        print(f" Roll Number   : {self.roll_no}")
        print(f" Name          : {self.name}")
        print(f" Age           : {self.age}")
        print(f" Course        : {self.course}")
        print("-" * 50)
        print(f" Marks         : {marks_text}")
        print(f" Total Marks   : {self.calculate_total():g}")
        print(f" Average Marks : {self.calculate_average():.2f}")
        print(f" Grade         : {self.calculate_grade()}")
        print("=" * 50)

    # object ko dictionary me badalta hai taaki json me save ho sake
    def to_dict(self):
        return {
            "roll_no": self.roll_no,
            "name": self.name,
            "age": self.age,
            "course": self.course,
            "marks": self.marks
        }

    # saved dictionary se wapas object banata hai
    @classmethod
    def from_dict(cls, data):
        return cls(
            roll_no=data["roll_no"],
            name=data["name"],
            age=data["age"],
            course=data["course"],
            marks=data["marks"]
        )


class StudentManagementSystem:

    def __init__(self):
        self.students = {}
        self.load_students()

    def load_students(self):
        if not os.path.exists(FILE_NAME):
            return

        try:
            with open(FILE_NAME, "r", encoding="utf-8") as file:
                data = json.load(file)
                loaded = {}
                for roll_no, s_data in data.items():
                    loaded[roll_no] = Student.from_dict(s_data)
                self.students = loaded

        except (json.JSONDecodeError, KeyError, TypeError, AttributeError):
            # file kharab hai to uski backup bana lete hain, data delete nahi karte
            backup_name = FILE_NAME + ".bak"
            print(f"\n[!] '{FILE_NAME}' is damaged or in wrong format.")
            try:
                os.replace(FILE_NAME, backup_name)
                print(f"    Backup saved as '{backup_name}'. Starting with empty records.")
            except OSError:
                print("    Starting with empty records.")
            self.students = {}

        except OSError:
            print(f"\n[!] Could not read '{FILE_NAME}'. Starting with empty records.")
            self.students = {}

    def save_students(self):
        # pehle temp file me likhte hain, phir replace karte hain
        # isse save ke beech me error aaye to purana data safe rehta hai
        temp_name = FILE_NAME + ".tmp"
        try:
            json_data = {roll: s.to_dict() for roll, s in self.students.items()}
            with open(temp_name, "w", encoding="utf-8") as file:
                json.dump(json_data, file, indent=4, ensure_ascii=False)
            os.replace(temp_name, FILE_NAME)
            return True
        except OSError:
            print("\n[Error] Could not save data!")
            return False

    def add_student(self):
        print("\n--- Add New Student ---")
        roll_no = get_text("Roll Number: ")

        if roll_no in self.students:
            print(f"[!] Roll Number {roll_no} already exists.")
            return

        name = get_text("Student Name: ")
        course = get_text("Course (e.g. BCA, B.Tech): ")
        age = get_int("Age (5-100): ", 5, 100)
        marks = get_marks()

        self.students[roll_no] = Student(roll_no, name, age, course, marks)

        if self.save_students():
            print("\n[✓] Student added and saved.")

    def search_student(self):
        print("\n--- Search Student ---")
        roll_no = get_text("Roll Number: ")

        student = self.students.get(roll_no)
        if student:
            student.display_report()
        else:
            print(f"[!] Roll Number {roll_no} not found.")

    def display_all_students(self):
        if not self.students:
            print("\n[!] No records found.")
            return

        print(f"\nTotal students: {len(self.students)}")
        for student in self.students.values():
            student.display_report()

    def update_student(self):
        print("\n--- Update Student ---")
        roll_no = get_text("Roll Number: ")

        student = self.students.get(roll_no)
        if not student:
            print(f"[!] Roll Number {roll_no} not found.")
            return

        print(f"\nUpdating: {student.name} (Roll No. {roll_no})")
        print("1. Name")
        print("2. Age")
        print("3. Course")
        print("4. Marks")
        print("5. Cancel")

        choice = get_int("What do you want to update? (1-5): ", 1, 5)

        if choice == 1:
            student.name = get_text("New name: ")
        elif choice == 2:
            student.age = get_int("New age (5-100): ", 5, 100)
        elif choice == 3:
            student.course = get_text("New course: ")
        elif choice == 4:
            student.marks = get_marks()
        else:
            print("\nUpdate cancelled.")
            return

        if self.save_students():
            print("\n[✓] Record updated.")

    def delete_student(self):
        print("\n--- Delete Student ---")
        roll_no = get_text("Roll Number: ")

        student = self.students.get(roll_no)
        if not student:
            print(f"[!] Roll Number {roll_no} not found.")
            return

        confirm = input(f"Are you sure you want to delete '{student.name}'? (y/n): ").strip().lower()

        if confirm == "y":
            del self.students[roll_no]
            if self.save_students():
                print("\n[✓] Record deleted.")
        else:
            print("\nDeletion cancelled.")


def show_menu():
    print("\n==========================================")
    print("       STUDENT MANAGEMENT SYSTEM")
    print("==========================================")
    print("1. Add New Student")
    print("2. Search Student")
    print("3. Display All Students")
    print("4. Update Student")
    print("5. Delete Student")
    print("6. Exit")
    print("==========================================")


def main():
    system = StudentManagementSystem()

    try:
        while True:
            show_menu()
            choice = input("Choose an option (1-6): ").strip()

            if choice == "1":
                system.add_student()
            elif choice == "2":
                system.search_student()
            elif choice == "3":
                system.display_all_students()
            elif choice == "4":
                system.update_student()
            elif choice == "5":
                system.delete_student()
            elif choice == "6":
                print("\nExiting program...")
                break
            else:
                print("\n[!] Invalid option. Choose between 1 and 6.")

    except (KeyboardInterrupt, EOFError):
        print("\n\nProgram closed.")


if __name__ == "__main__":
    main()


       STUDENT MANAGEMENT SYSTEM
1. Add New Student
2. Search Student
3. Display All Students
4. Update Student
5. Delete Student
6. Exit
Choose an option (1-6): 1

--- Add New Student ---
Roll Number: 101
Student Name: kratagya pandey
Course (e.g. BCA, B.Tech): MCA
Age (5-100): 21
Number of subjects (1-15): 3
  Subject 1 marks (0-100): 58
  Subject 2 marks (0-100): 47
  Subject 3 marks (0-100): 78

[✓] Student added and saved.

       STUDENT MANAGEMENT SYSTEM
1. Add New Student
2. Search Student
3. Display All Students
4. Update Student
5. Delete Student
6. Exit
Choose an option (1-6): 3

Total students: 1

            STUDENT REPORT CARD
 Roll Number   : 101
 Name          : kratagya pandey
 Age           : 21
 Course        : MCA
--------------------------------------------------
 Marks         : 58, 47, 78
 Total Marks   : 183
 Average Marks : 61.00
 Grade         : C

       STUDENT MANAGEMENT SYSTEM
1. Add New Student
2. Search Student
3. Display All Students
4. Update Student
